# 00 setup

Run once per Colab session. Runtime -> Change runtime type -> T4 GPU.

Code is pulled from GitHub with a single-repo fine-grained token (Contents: read and write, 30-day expiry) stored in Colab Secrets as `GH_TOKEN`. The same token lets the last cell push small result files back. Checkpoints and data stay on Drive.

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q -e /content/transaction-fm
%cd /content/transaction-fm
import tfm
print("tfm ok")

## Check the GPU and the Drive layout

In [ ]:
import os, torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
for d in ["data/raw", "data/processed", "pretrain", "finetune", "xgb", "results"]:
    p = f"/content/drive/MyDrive/tfm/{d}"
    os.makedirs(p, exist_ok=True)
    print(f"{d:16s}", sorted(os.listdir(p))[:10])

`data/raw` should list the two Sparkov csv files (`fraudTrain.csv`, `fraudTest.csv`) before step 1.

## Push results back to GitHub

Run after a step has written json or png into `MyDrive/tfm/results/`. Only small files are copied; checkpoints never leave Drive.

In [ ]:
import shutil, glob, os
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob("/content/drive/MyDrive/tfm/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob("/content/drive/MyDrive/tfm/results/*.png"):
    shutil.copy(f, "results/figures/")
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results && git commit -q -m "results from Colab" && git push -q
!git log --oneline | head -1

## End of session

In [ ]:
drive.flush_and_unmount()